In [0]:
# Databricks Notebook: 01_Bronze_Ingest
# Bronze 层：source → bronze 原样落地 + 摄取元数据
# 幂等策略：DELETE 当天 ingest_date → APPEND（静态表 DELETE 全量 → APPEND）
# 约定：新 Delta 表用 Liquid Clustering (CLUSTER BY)

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime

today = datetime.now().strftime("%Y-%m-%d")
print(f"📅 Bronze 摄取日期: {today}")

# =====================================================
# 1. 表清单（静态主数据 vs 每日事实数据）
# =====================================================
static_tables = ["raw_route_def", "raw_products"]          # 全量覆盖
daily_tables = ["raw_mes", "raw_mes_moves", "raw_lot_events",
                "raw_cp_bins", "raw_equip_state",
                "raw_quality", "raw_sensors"]              # 按天幂等

def ensure_bronze_table(name, cluster_col):
    """首次运行时建空表并启用 Liquid Clustering"""
    if not spark.catalog.tableExists(f"bronze.{name}"):
        cols = ", ".join(f"`{f.name}` {f.dataType.simpleString()}"
                         for f in spark.table(f"source.{name}").schema.fields)
        spark.sql(f"""
            CREATE TABLE bronze.{name} ({cols}, ingest_date STRING, ingest_ts TIMESTAMP)
            CLUSTER BY ({cluster_col})
        """)
        print(f"🆕 已创建 bronze.{name} (CLUSTER BY {cluster_col})")

# =====================================================
# 2. 静态主数据：DELETE 全量 → APPEND
# =====================================================
print("=" * 50)
print("静态主数据（全量覆盖）")
print("=" * 50)
for t in static_tables:
    ensure_bronze_table(t, "ingest_date")
    spark.sql(f"DELETE FROM bronze.{t}")
    (spark.table(f"source.{t}")
        .withColumn("ingest_date", F.lit(today))
        .withColumn("ingest_ts", F.current_timestamp())
        .write.mode("append").format("delta").saveAsTable(f"bronze.{t}"))
    print(f"✅ bronze.{t}: {spark.table(f'bronze.{t}').count()} 行（全量）")

# =====================================================
# 3. 每日事实表：DELETE 当天 → APPEND（幂等核心）
# =====================================================
print("=" * 50)
print("每日事实数据（按天幂等）")
print("=" * 50)
for t in daily_tables:
    ensure_bronze_table(t, "ingest_date")
    # 先清当天旧数据（重跑安全）
    spark.sql(f"DELETE FROM bronze.{t} WHERE ingest_date = '{today}'")
    (spark.table(f"source.{t}")
        .withColumn("ingest_date", F.lit(today))
        .withColumn("ingest_ts", F.current_timestamp())
        .write.mode("append").format("delta").saveAsTable(f"bronze.{t}"))
    cnt = spark.sql(f"SELECT COUNT(*) FROM bronze.{t} WHERE ingest_date = '{today}'").first()[0]
    print(f"✅ bronze.{t}: 当天 +{cnt} 行")

# =====================================================
# 4. 验证
# =====================================================
print("=" * 50)
print("Bronze 层当天行数汇总")
print("=" * 50)
for t in static_tables + daily_tables:
    cnt = spark.sql(f"SELECT COUNT(*) FROM bronze.{t}").first()[0]
    print(f"  bronze.{t}: {cnt}")
print(f"\n🎉 Bronze 摄取完成！重跑本 Notebook 结果一致（幂等）")


📅 Bronze 摄取日期: 2026-09-03
静态主数据（全量覆盖）
✅ bronze.raw_route_def: 810 行（全量）
✅ bronze.raw_products: 5 行（全量）
每日事实数据（按天幂等）
✅ bronze.raw_mes: 当天 +210 行
✅ bronze.raw_mes_moves: 当天 +1216 行
✅ bronze.raw_lot_events: 当天 +21 行
✅ bronze.raw_cp_bins: 当天 +0 行
✅ bronze.raw_equip_state: 当天 +282 行
✅ bronze.raw_quality: 当天 +200 行
✅ bronze.raw_sensors: 当天 +144 行
Bronze 层当天行数汇总
  bronze.raw_route_def: 810
  bronze.raw_products: 5
  bronze.raw_mes: 210
  bronze.raw_mes_moves: 1216
  bronze.raw_lot_events: 21
  bronze.raw_cp_bins: 0
  bronze.raw_equip_state: 282
  bronze.raw_quality: 200
  bronze.raw_sensors: 144

🎉 Bronze 摄取完成！重跑本 Notebook 结果一致（幂等）
